---
title: LLM-based Music Annotation
authors:
  - name: SeungHeon Doh
---

The [annotation page](./annotation.md) argued that one audio-language model can fill many columns of a music database, with the instruction choosing the column. This notebook does that on a real dataset. MagnaTagATune {cite}`law_magnatagatune_2009` labels short clips with tags collected through an online game, and LP-MusicCaps {cite}`doh2023lp` turned those tags into captions. The tags are what a database would be built from, and they are uneven: one clip has eighteen tags, many of them synonyms, another has one. Many clips have no instrument tag at all, and nothing records where the music would be used.

We take 50 clips, sort their tags into columns, and ask an audio LLM to expand them: more genre, instrument, and mood tags for every clip, and a new column for where the music would be used. We stay with semantic tags, the words a listener would search with. Tempo and key are left out, because signal-processing tools measure them better than a language model guesses them. The model is Audex-2B {cite}`kong_audex_2026`, a general audio-text model small enough to run on a laptop.

## What we will build

1. Load 50 clips with their tags and captions.
2. Sort the raw tags into columns: genre, instrument, mood, vocal. This shows how thin the tags are.
3. Ask the audio LLM to describe each clip, then turn the description into tags for every column, including one column MagnaTagATune never had: *usage*, the situations the music would suit.
4. Expand the table: keep the players' tags, add the model's tags next to them, and record which tags the model added.




*The three steps on one clip. Gray values come from the tags, amber values are generated, and dashed cells are empty. The model's values here are illustrative.*


## Prerequisites

Steps 1 to 3 need only a CPU. From Step 4 on, the notebook loads Audex-2B: a 2B-parameter LLM with a Whisper-style audio encoder, about 6 GB to download. The code picks a CUDA GPU, Apple silicon, or the CPU, in that order. The outputs shown below come from a run on an Apple M3 laptop (16 GB of memory, no CUDA GPU), which took about 23 minutes for the 50 clips.

Audex-2B is a general audio-text model. It was not trained specifically for music, so a larger music model such as Music Flamingo {cite}`ghosh_musicflamingo_2026` should give better values; to try one, replace `ask` in Step 4. Audex-2B is released for non-commercial use only.


## Step 1: Setting up the environment

In [1]:
%pip install -q "transformers>=5.5" accelerate datasets librosa soundfile pandas huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json, os, re, textwrap
from collections import defaultdict

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from IPython.display import Audio, display

pd.set_option("display.max_colwidth", 80)

## Step 2: Loading the clips

The dataset on the Hub, `mulab-mir/lp-music-caps-magnatagatune-3k`, holds 3,000 MagnaTagATune clips in its training split. Each row has the clip as a 10-second waveform at 22,050 Hz, the MagnaTagATune tags, and four captions LP-MusicCaps generated from those tags. We stream the first 50 rows, which avoids downloading the full 2.6 GB, and write each clip to a WAV file for the model.

In [2]:
from datasets import load_dataset

N_TRACKS = 50
AUDIO_DIR = "mtat_audio"
os.makedirs(AUDIO_DIR, exist_ok=True)

stream = load_dataset("mulab-mir/lp-music-caps-magnatagatune-3k", split="train", streaming=True)

rows = []
for row in stream.take(N_TRACKS):
    audio_path = os.path.join(AUDIO_DIR, f"{row['track_id']}.wav")
    sf.write(audio_path, np.asarray(row["audio"]["array"], dtype=np.float32), row["audio"]["sampling_rate"])
    rows.append({"track_id": row["track_id"], "audio_path": audio_path,
                 "tags": row["tags"], "caption": row["texts"][0]})

mtat = pd.DataFrame(rows)
mtat["n_tags"] = mtat.tags.str.len()
mtat[["track_id", "n_tags", "tags"]].head(10)

,track_id,n_tags,tags
0,47315,2,"[rap, hip hop]"
1,4754,9,"[classical, ambient, synth, space, organ, no vocal, dark, no vocals, instrum..."
2,54505,3,"[male singer, punk, rock]"
3,3878,4,"[no voice, sitar, indian, india]"
4,36621,4,"[female voice, girl, vocal, woman]"
5,30702,15,"[no voice, horns, jazz, trumpet, drums, electronic, no vocal, no vocals, hor..."
6,28694,3,"[wind, no vocal, no vocals]"
7,38393,8,"[clarinet, classical, solo, wind, oboe, slow, flute, soft]"
8,29864,15,"[vocals, female, quiet, female voice, low, mellow, slow, vocal, woman, singi..."
9,31872,2,"[no piano, sax]"


The number of tags per clip ranges widely. Here is a clip with many, and what LP-MusicCaps wrote from them. The caption is fluent, but it can only restate the tags it was given.

In [3]:
def play(audio_path):
    """A player for one clip. The clip is embedded as MP3, which keeps the notebook small."""
    mp3_path = audio_path.replace(".wav", ".mp3")
    if not os.path.exists(mp3_path):
        sf.write(mp3_path, *librosa.load(audio_path, sr=None), format="MP3")
    return Audio(mp3_path, embed=True)


print(mtat.n_tags.describe()[["min", "50%", "max"]].to_string())

example = mtat.loc[mtat.n_tags.idxmax()]
display(play(example.audio_path))
print(example.track_id, example.tags)
print(textwrap.fill(example.caption, 100))

min     1.0
50%     4.0
max    18.0


45580 ['singer', 'sitar', 'male vocal', 'soft rock', 'guitar', 'voice', 'male voice', 'loud', 'drums', 'male singer', 'modern', 'man', 'pop', 'male vocals', 'male', 'vocal', 'rock', 'singing']
This modern soft rock/pop song features a male singer with a captivating voice accompanied by a
sitar, guitar, and drums, creating a captivating blend of male vocals and traditional rock elements
that build from soft to loud, with the male voice helping tell the story.


## Step 3: Sorting the tags into columns

A database wants columns, not a bag of tags. We map each raw tag to a column and a canonical value. The map does three kinds of cleanup. It merges synonyms (*woman*, *female voice*, and *female vocals* all become *female vocal*), fixes typos (*clasical*), and drops negative tags such as *no piano* or *not opera*, which say what is absent and are not values of any column. The tags that say there are no vocals become the vocal value *instrumental*. Tags that fit no column (*solo*, *beat*, *modern*) are set aside, and so are the tempo words (*slow*, *fast*).

In [4]:
TAG_MAP = {
    "genre": {
        "hip hop": ["hip hop", "rap"], "classical": ["classical", "clasical", "classic"],
        "rock": ["rock"], "soft rock": ["soft rock"], "hard rock": ["hard rock"],
        "metal": ["metal", "heavy metal"], "punk": ["punk"], "pop": ["pop"], "jazz": ["jazz", "jazzy"],
        "electronic": ["electronic"], "techno": ["techno"], "dance": ["dance", "disco"],
        "ambient": ["ambient", "new age", "drone"], "folk": ["folk"], "country": ["country"],
        "celtic": ["celtic"], "reggae": ["reggae"], "indian": ["indian", "india", "eastern"],
        "medieval": ["medieval"], "opera": ["opera", "operatic"],
    },
    "instrument": {
        "guitar": ["guitar"], "acoustic guitar": ["acoustic guitar", "acoustic"], "banjo": ["banjo"],
        "sitar": ["sitar"], "piano": ["piano"], "harpsichord": ["harpsichord"], "organ": ["organ"],
        "synthesizer": ["synth"], "strings": ["strings"], "violin": ["violin"], "harp": ["harp"],
        "flute": ["flute", "flutes"], "woodwinds": ["wind", "woodwind"], "oboe": ["oboe"],
        "clarinet": ["clarinet"], "saxophone": ["sax"], "trumpet": ["trumpet"], "horns": ["horn", "horns"],
        "drums": ["drums", "drum"], "percussion": ["percussion", "bongos"], "bells": ["bells", "chimes"],
    },
    "mood_theme": {
        "calm": ["calm", "mellow"], "soft": ["soft", "quiet"], "loud": ["loud", "heavy", "hard"],
        "dark": ["dark"], "happy": ["happy"], "weird": ["weird"], "spacey": ["space"],
    },
    "vocal": {
        "female vocal": ["female", "woman", "women", "girl", "female voice", "female vocal",
                         "female vocals", "female singer", "woman singing"],
        "male vocal": ["male", "man", "male voice", "male vocal", "male vocals", "male singer", "man singing"],
        "choir": ["choir"], "duet": ["duet"],
        "vocals": ["vocal", "vocals", "voice", "voices", "singing", "singer"],
        "instrumental": ["no vocal", "no vocals", "no voice", "instrumental"],
    },
}
COLUMNS = list(TAG_MAP)
LOOKUP = {raw: (col, value) for col, values in TAG_MAP.items() for value, raws in values.items() for raw in raws}


def sort_tags(tags):
    """Return {column: [canonical values]} and the raw tags that fit no column."""
    columns, unsorted = {col: [] for col in COLUMNS}, []
    for tag in tags:
        if tag in LOOKUP:
            col, value = LOOKUP[tag]
            if value not in columns[col]:
                columns[col].append(value)
        elif not tag.startswith(("no ", "not ")):
            unsorted.append(tag)
    if len(columns["vocal"]) > 1 and "vocals" in columns["vocal"]:
        columns["vocal"].remove("vocals")  # "vocals" adds nothing next to "female vocal"
    return columns, unsorted


sorted_tags = mtat.tags.apply(sort_tags)
tag_table = pd.DataFrame([c for c, _ in sorted_tags], index=mtat.track_id)
print("set aside:", sorted({t for _, u in sorted_tags for t in u}))
tag_table.head(10)

set aside: ['beat', 'beats', 'fast', 'foreign', 'low', 'modern', 'slow', 'solo', 'upbeat']


,genre,instrument,mood_theme,vocal
track_id,,,,
47315,[hip hop],[],[],[]
4754,"[classical, ambient]","[synthesizer, organ]","[spacey, dark]",[instrumental]
54505,"[punk, rock]",[],[],[male vocal]
3878,[indian],[sitar],[],[instrumental]
36621,[],[],[],[female vocal]
30702,"[jazz, electronic, pop]","[horns, trumpet, drums, saxophone]",[],[instrumental]
28694,[],[woodwinds],[],[instrumental]
38393,[classical],"[clarinet, woodwinds, oboe, flute]",[soft],[]
29864,[],[],"[soft, calm]","[female vocal, choir]"


The example clip's tags collapse to a handful of values. More important is what the table does not have. The share of clips with each column empty:

In [5]:
print(tag_table.loc[example.track_id].to_dict())
empty_before = tag_table.map(len).eq(0).mean().rename("empty (tags only)")
empty_before.to_frame().style.format("{:.0%}")

{'genre': ['soft rock', 'pop', 'rock'], 'instrument': ['sitar', 'guitar', 'drums'], 'mood_theme': ['loud'], 'vocal': ['male vocal']}


,empty (tags only)
genre,28%
instrument,24%
mood_theme,70%
vocal,48%


An empty cell in MagnaTagATune does not mean the attribute is absent. Players tagged what struck them in the clip, so a missing *vocal* value can mean an instrumental track or a singer nobody mentioned. The game also never asked where the music would be used. The model will fill these gaps and add to the cells that already have a value.

## Step 4: Loading the model

`ask(audio_path, instruction)` sends one clip and one instruction and returns the model's text. With `audio_path=None` it sends the instruction alone, which we use in Step 5 to work on text. Decoding is greedy, so a rerun gives the same answer.

The model ships its own code, so it is loaded with `trust_remote_code=True` from the `checkpoint_folder_full` folder of the repository. The steps follow the repository's inference script. The clip is resampled to 16 kHz and cut into 30-second windows, a Whisper feature extractor turns each window into a mel spectrogram, and each window takes 750 `<so_embedding>` placeholders in the prompt, which the model replaces with audio embeddings. The assistant turn starts with `<think></think>`, the prefix the model card gives for audio tasks without reasoning, or with `</think>` for a text-only request.

In [6]:
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoFeatureExtractor, AutoModelForCausalLM, AutoTokenizer

repo = snapshot_download("nvidia/Nemotron-Labs-Audex-2B", allow_patterns=["checkpoint_folder_full/*"])
CKPT = os.path.join(repo, "checkpoint_folder_full")
if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda:0", torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif torch.backends.mps.is_available():  # Apple silicon
    DEVICE, DTYPE = "mps", torch.float16
else:
    DEVICE, DTYPE = "cpu", torch.float32

tokenizer = AutoTokenizer.from_pretrained(CKPT, trust_remote_code=True)
config = AutoConfig.from_pretrained(CKPT, trust_remote_code=True)
feature_extractor = AutoFeatureExtractor.from_pretrained(os.path.join(CKPT, "audio_preprocessor"))
model = AutoModelForCausalLM.from_pretrained(
    CKPT, trust_remote_code=True, dtype=DTYPE, device_map={"": DEVICE}
).eval()
print(f"device: {DEVICE}, dtype: {DTYPE}")

SR, WINDOW = 16000, 30  # the audio encoder reads 16 kHz audio in 30-second windows
TOKENS_PER_WINDOW = int(getattr(config, "sound_embedding_size", 750))
IM_END = tokenizer.convert_tokens_to_ids("<|im_end|>")

def ask(audio_path, instruction, max_new_tokens=400):
    extra = {}
    if audio_path is None:  # text only
        prompt = f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n</think>\n"
    else:
        audio, _ = librosa.load(audio_path, sr=SR, mono=True)
        n_windows = max(1, int(np.ceil(len(audio) / (SR * WINDOW))))
        audio = np.pad(audio, (0, n_windows * SR * WINDOW - len(audio))).astype(np.float32)
        windows = list(audio.reshape(n_windows, SR * WINDOW))
        input_features = feature_extractor(
            windows, sampling_rate=SR, return_tensors="pt", padding="max_length", return_attention_mask=False
        ).input_features  # (n_windows, 128, 3000)
        extra["input_features"] = input_features.to(model.device)
        sound = "<so_start>" + "<so_embedding>" * (n_windows * TOKENS_PER_WINDOW) + "<so_end>"
        prompt = f"<|im_start|>user\n{sound}\n{instruction}<|im_end|>\n<|im_start|>assistant\n<think></think>"
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            eos_token_id=IM_END,
            pad_token_id=tokenizer.pad_token_id or 0,
            **extra,
        )
    text = tokenizer.decode(outputs[0, inputs.input_ids.shape[1]:], skip_special_tokens=False)
    return text.split("<|im_end|>")[0].replace("</think>", "").strip()


device: mps, dtype: torch.float16


A quick check on the example clip before running all 50:

In [8]:
print(ask(example.audio_path, "Describe this music in two sentences."))


This is an alternative rock song with a mellow, melodic vibe. The male vocalist sings melodically with a guitar accompaniment. The bass guitar is prominent in the mix. The song is played at a medium tempo with a 4-beat rhythm. The chord progression in this song is E, A, C#m, E. The key of this song is E major. The song is played at the pace of Vivace.


## Step 5: Describing each clip, then proposing values

We fill the columns in two stages. First the model listens to the clip and describes it in words. Then the description alone, without the audio, is turned into a row. Asking a small model for every column at once from the audio gave worse values in our runs: for an ambient clip it wrote *indie* and *drums*, while its own description of the same clip said *ambient* with a synth pad. Writing a description is close to what the model was trained to do, and filling a form from a description is a text task.

Each column is one line in a dictionary: a name and a definition in words. The second prompt is built from the dictionary and asks for JSON, so the answer is a row we can parse. The first four keys are the columns from Step 3. *Usage* is new: MagnaTagATune never had it. Adding a column means adding a line. *Vocal* is the one closed column: the prompt lists its four allowed values, and we discard anything else.

:::{figure} img/annotation-metadata-llm.png
:label: fig-demo-metadata-imputation
:width: 90%
:alt: Baseline: a multimodal LLM turns audio into a caption, and an optional text LLM extracts genre, mood, and instrument fields from the caption. Proposed: a multimodal LLM reads the audio and a partly missing metadata row and outputs the full set of fields, and an optional text LLM turns the fields into a caption.

Two orders for a multi-stage annotation pipeline: audio to caption to tags, against audio to tags to caption. Step 5 of this notebook follows the first order, with one model in both roles: it writes the description from the audio and then extracts the fields from the description. Figure from {cite}`bukey_metadata_2026`.
:::

:::{figure} ../assets/annotation-sequence.svg
:label: fig-demo-annotation-sequence
:alt: Three rows showing annotation as one sequence model. Tagging: audio and the instruction "Annotate genre, mood, instruments" are given, and the tags are generated. Captioning: audio and the instruction "Write a caption" are given, and the caption is generated. Imputation: audio, an instruction, and partly filled metadata are given, and the missing values are generated.

One model, many outputs. The audio and the instruction are given, and the model generates the answer token by token. We use the captioning row first, and then a text-only request for the columns.
:::

In [9]:
VOCAL = ["female vocal", "male vocal", "choir", "instrumental"]
PROPOSE = {
    "genre": "a list of one to three genres or styles",
    "instrument": "a list of the instruments in the description",
    "mood_theme": "a list of up to four mood or theme words",
    "vocal": "a list from " + ", ".join(f'"{v}"' for v in VOCAL),
    "usage": "a list of two or three situations this music would suit",
}

DESCRIBE = ("Describe this music in detail: its genre and style, the instruments you hear, the mood, "
            "whether there are vocals, and where it would be used.")

TO_ROW = (
    "A description of a music track:\n{description}\n\n"
    "Turn the description into a catalog entry. Answer with one JSON object and nothing else, using these keys:\n"
    + "\n".join(f'"{name}": {definition},' for name, definition in PROPOSE.items())
)
print(DESCRIBE, TO_ROW, sep="\n\n")


Describe this music in detail: its genre and style, the instruments you hear, the mood, whether there are vocals, and where it would be used.

A description of a music track:
{description}

Turn the description into a catalog entry. Answer with one JSON object and nothing else, using these keys:
"genre": a list of one to three genres or styles,
"instrument": a list of the instruments in the description,
"mood_theme": a list of up to four mood or theme words,
"vocal": a list from "female vocal", "male vocal", "choir", "instrumental",
"usage": a list of two or three situations this music would suit,


Two requests per clip: one with the audio, one with the description. The model's words go through the same map as the players' tags, so that *synth* and *synthesizer* end up as one value.

In [10]:
def parse_row(text):
    """Return the first JSON object in the model's answer, or None."""
    match = re.search(r"\{.*\}", text, flags=re.S)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError:
        return None


def as_tags(col, value):
    """Lowercase the model's values, map known synonyms to the canonical tag, and drop duplicates."""
    values = value if isinstance(value, list) else [value] if value else []
    tags = []
    for v in values:
        v = str(v).strip().lower().replace("-", " ")
        if LOOKUP.get(v, (None,))[0] == col:
            v = LOOKUP[v][1]
        if v and v not in tags and (col != "vocal" or v in VOCAL):
            tags.append(v)
    return tags


descriptions, proposals = {}, {}
for track in mtat.itertuples():
    descriptions[track.track_id] = ask(track.audio_path, DESCRIBE, max_new_tokens=200)
    answer = ask(None, TO_ROW.replace("{description}", descriptions[track.track_id]))
    proposals[track.track_id] = parse_row(answer) or {}
print(sum(not row for row in proposals.values()), "of", len(proposals), "answers could not be parsed")

proposed = pd.DataFrame(
    [{col: as_tags(col, proposals[tid].get(col)) for col in PROPOSE} for tid in mtat.track_id],
    index=mtat.track_id,
)
proposed.head(10)


0 of 50 answers could not be parsed


,genre,instrument,mood_theme,vocal,usage
track_id,,,,,
47315,[hip hop],"[drums, bass, keyboard]","[energetic, youthful, groovy]",[male vocal],"[youthful events, urban lifestyle videos, workout playlists]"
4754,[ambient],"[synthesizers, electronic instruments]","[peaceful, relaxing, introspective, contemplative]",[instrumental],"[yoga studios, meditation centers, spa facilities]"
54505,[hardcore punk],"[electric guitar, bass guitar, drums, keyboard]","[aggressive, energetic, fast paced, intense]",[male vocal],"[concerts, underground music scenes, rock festivals]"
3878,"[indian instrumental, indian]",[sitar],"[meditative, ceremonial, tranquil]",[instrumental],"[meditation, traditional ceremonies, background ambiance]"
36621,"[jazz, hip hop]","[bass, hi hats, kick, snare, saxophone]","[groovy, soulful, chillout, calm]",[instrumental],"[chillout session, relaxation, background music]"
30702,"[jazz, brass band]","[brass, kick drum, snare drum]","[calm, groovy, laid back]",[instrumental],"[background music for a casual video, ambient background in a low quality re..."
28694,[ambient],[flute],"[relaxing, ethereal, meditative]",[instrumental],"[meditation center, spa, yoga studio]"
38393,[classical],"[clarinet, flute, violin, cello, bassoon]","[calm, peaceful]",[instrumental],"[concert hall performance, wedding ceremony, meditation session]"
29864,[ambient],"[keyboard, bass, choir]","[calm, tranquility, peaceful]",[choir],"[meditation, background study, relaxation]"


For the example clip, the model's description, and then the tags and the model's values side by side:

In [11]:
print(textwrap.fill(descriptions[example.track_id], 100))
pd.DataFrame({
    "tags": {**tag_table.loc[example.track_id].to_dict(), "usage": []},
    "model": proposed.loc[example.track_id].to_dict(),
})


This is a live performance of an alternative rock song. The song is medium tempo with a repetitive
acoustic guitar accompaniment, steady drumming and a male vocalist singing melodically. The song is
emotional and passionate. This song is in the key of E major. The chord sequence is E, A, C#m, E.
The beat is 4. This song goes at 166.0 beats per minute.


,tags,model
genre,"[soft rock, pop, rock]",[alternative rock]
instrument,"[sitar, guitar, drums]","[acoustic guitar, drums, male vocalist]"
mood_theme,[loud],"[emotional, passionate]"
vocal,[male vocal],[male vocal]
usage,[],"[live performance, concert setting, emotional storytelling]"


The description also states a key, a chord sequence, and a tempo in beats per minute. We did not ask for them, and nothing here checks them, which is one reason this notebook keeps to semantic tags.

The vocabulary is open: *instrument* holds whatever the model heard, not a fixed label set, which is the point of the [autoregressive generation](./annotation.md#sec-annotation-llm) capability.

## Step 6: Expanding the tags

The tags and the model now give two sets of values for each cell. For the open columns (genre, instrument, mood) we keep the players' tags and add the model's tags that are not there yet. *Vocal* is a closed column where two different answers contradict each other, so the tags win and the model fills only the empty cells. *Usage* comes from the model alone. We also keep the list of tags the model added, so a later user can filter them out or replace the rule.


In [12]:
OPEN = ["genre", "instrument", "mood_theme"]

table, added = pd.DataFrame(index=mtat.track_id), pd.DataFrame(index=mtat.track_id)
for col in OPEN:
    added[col] = [[v for v in new if v not in tags] for tags, new in zip(tag_table[col], proposed[col])]
    table[col] = [tags + extra for tags, extra in zip(tag_table[col], added[col])]
added["vocal"] = [[] if tags else new for tags, new in zip(tag_table["vocal"], proposed["vocal"])]
table["vocal"] = [tags or new for tags, new in zip(tag_table["vocal"], proposed["vocal"])]
table["usage"] = added["usage"] = proposed["usage"]

pd.DataFrame({
    "tags per clip (players)": tag_table.map(len).mean(),
    "tags per clip (expanded)": table.map(len).mean(),
}).reindex(table.columns).fillna(0).round(1)


,tags per clip (players),tags per clip (expanded)
genre,1.3,2.5
instrument,1.4,3.8
mood_theme,0.4,3.6
vocal,0.6,1.0
usage,0.0,3.0


The two sources do not always agree. The check below counts the clips where the players' tags and the model disagree on whether anyone sings, the one question in this table with a yes-or-no answer.

In [13]:
SINGING = {"female vocal", "male vocal", "choir", "duet", "vocals"}
sings = lambda values: bool(SINGING & set(values))

both = [tid for tid in mtat.track_id if tag_table.vocal[tid] and proposed.vocal[tid]]
conflict = [tid for tid in both if sings(tag_table.vocal[tid]) != sings(proposed.vocal[tid])]
print(f"{len(conflict)} of {len(both)} clips with a vocal value from both sources disagree on whether anyone sings")
pd.DataFrame({"tags": tag_table.vocal[conflict], "model": proposed.vocal[conflict],
              "description": [textwrap.shorten(descriptions[tid], 110) for tid in conflict]}).head(5)


2 of 25 clips with a vocal value from both sources disagree on whether anyone sings


,tags,model,description
track_id,,,
36621,[female vocal],[instrumental],"This music is a fusion of jazz and hip hop. It features a groovy bassline, s..."
32926,[female vocal],[instrumental],"This music is an instrumental. The tempo is medium with a keyboard harmony, ..."


## Results

The expanded table: the players' tags, the model's tags added to them, and the new *usage* column.


In [14]:
table.join(added.add_suffix("_added")).reset_index().to_json("mtat_expanded.jsonl", orient="records", lines=True)
table.head(10)


,genre,instrument,mood_theme,vocal,usage
track_id,,,,,
47315,[hip hop],"[drums, bass, keyboard]","[energetic, youthful, groovy]",[male vocal],"[youthful events, urban lifestyle videos, workout playlists]"
4754,"[classical, ambient]","[synthesizer, organ, synthesizers, electronic instruments]","[spacey, dark, peaceful, relaxing, introspective, contemplative]",[instrumental],"[yoga studios, meditation centers, spa facilities]"
54505,"[punk, rock, hardcore punk]","[electric guitar, bass guitar, drums, keyboard]","[aggressive, energetic, fast paced, intense]",[male vocal],"[concerts, underground music scenes, rock festivals]"
3878,"[indian, indian instrumental]",[sitar],"[meditative, ceremonial, tranquil]",[instrumental],"[meditation, traditional ceremonies, background ambiance]"
36621,"[jazz, hip hop]","[bass, hi hats, kick, snare, saxophone]","[groovy, soulful, chillout, calm]",[female vocal],"[chillout session, relaxation, background music]"
30702,"[jazz, electronic, pop, brass band]","[horns, trumpet, drums, saxophone, brass, kick drum, snare drum]","[calm, groovy, laid back]",[instrumental],"[background music for a casual video, ambient background in a low quality re..."
28694,[ambient],"[woodwinds, flute]","[relaxing, ethereal, meditative]",[instrumental],"[meditation center, spa, yoga studio]"
38393,[classical],"[clarinet, woodwinds, oboe, flute, violin, cello, bassoon]","[soft, calm, peaceful]",[instrumental],"[concert hall performance, wedding ceremony, meditation session]"
29864,[ambient],"[keyboard, bass, choir]","[soft, calm, tranquility, peaceful]","[female vocal, choir]","[meditation, background study, relaxation]"


Listen to the first six clips next to their tags. A `+` marks a tag the model added; the others came from the players.


In [ ]:
for track in mtat.head(6).itertuples():
    display(play(track.audio_path))
    for col in table.columns:
        new = added.at[track.track_id, col]
        kept = [v for v in table.at[track.track_id, col] if v not in new]
        print(f"{col:>10}: {', '.join(kept + ['+' + v for v in new])}")


     genre: hip hop
instrument: +drums, +bass, +keyboard
mood_theme: +energetic, +youthful, +groovy
     vocal: +male vocal
     usage: +youthful events, +urban lifestyle videos, +workout playlists


     genre: classical, ambient
instrument: synthesizer, organ, +synthesizers, +electronic instruments
mood_theme: spacey, dark, +peaceful, +relaxing, +introspective, +contemplative
     vocal: instrumental
     usage: +yoga studios, +meditation centers, +spa facilities


     genre: punk, rock, +hardcore punk
instrument: +electric guitar, +bass guitar, +drums, +keyboard
mood_theme: +aggressive, +energetic, +fast paced, +intense
     vocal: male vocal
     usage: +concerts, +underground music scenes, +rock festivals


     genre: indian, +indian instrumental
instrument: sitar
mood_theme: +meditative, +ceremonial, +tranquil
     vocal: instrumental
     usage: +meditation, +traditional ceremonies, +background ambiance


     genre: +jazz, +hip hop
instrument: +bass, +hi hats, +kick, +snare, +saxophone
mood_theme: +groovy, +soulful, +chillout, +calm
     vocal: female vocal
     usage: +chillout session, +relaxation, +background music


     genre: jazz, electronic, pop, +brass band
instrument: horns, trumpet, drums, saxophone, +brass, +kick drum, +snare drum
mood_theme: +calm, +groovy, +laid back
     vocal: instrumental
     usage: +background music for a casual video, +ambient background in a low quality recording, +drum and bass practice session


The share of clips with each column empty, before and after:

In [15]:
empty_after = table.map(len).eq(0).mean().rename("empty (expanded)")
pd.concat([empty_before, empty_after], axis=1).style.format("{:.0%}", na_rep="—")


,empty (tags only),empty (expanded)
genre,28%,0%
instrument,24%,0%
mood_theme,70%,0%
vocal,48%,0%
usage,—,0%


## Conclusion

We started from tags typed by players of a game: uneven in number, full of synonyms, and silent on use. A dictionary sorted them into columns, an audio LLM described each clip and turned the description into tags, and a simple rule added the model's tags to the players' while recording which ones were added. Mood went from 0.4 to 3.6 tags per clip, and no column is empty for any clip. *Usage* did not exist in the dataset; it cost one line in the dictionary.

More tags are not better tags, and the tables above show three reasons to check before using them.

- **The sources contradict each other.** For 2 of the 25 clips where both name the vocals, the players tagged a female voice and the model heard an instrumental. One of the two is wrong, and the table cannot say which.
- **The open vocabulary needs normalizing.** The map from Step 3 caught some synonyms, but *synthesizer* and *synthesizers* now sit side by side, and *male vocalist* appears as an instrument. The [evaluation section](./annotation.md#sec-annotation-evaluation) applies to every open column.
- **The new column has no reference.** *Usage* reads plausibly, but many clips get the same few situations (meditation, background music), and there is nothing to score it against.

To scale this up, batch several clips per `generate` call, and push the table back to the Hub with `datasets.Dataset.from_pandas(table).push_to_hub(...)`.


## Resources

- Audex-2B: [model card](https://huggingface.co/nvidia/Nemotron-Labs-Audex-2B), [paper](https://arxiv.org/abs/2607.05196)
- LP-MusicCaps MagnaTagATune 3k: [Hub](https://huggingface.co/datasets/mulab-mir/lp-music-caps-magnatagatune-3k), [LP-MusicCaps code](https://github.com/seungheondoh/lp-music-caps)
- MagnaTagATune: [dataset page](https://mirg.city.ac.uk/codeapps/the-magnatagatune-dataset)